# 密集子图搜索实验

本示例通过九章 SDK 的本地应用接口完成密集子图搜索。流程包括加载示例图、计算密度目标函数、比较随机搜索、模拟退火和采样增强搜索的效果。

## 1. 初始化环境

In [ ]:
from jiuzhang.local.applications import (
    dense_score,
    draw_graph,
    greedy_dense_subgraph,
    load_planted_dense_graph,
    load_sample_database,
    random_dense_search,
    sample_database_search,
    simulated_annealing_dense_search,
)

import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Sarasa UI SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

## 2. 设置实验参数

`TARGET_SIZE` 表示需要保留的节点数量；`ITERATIONS` 表示搜索算法评估的候选次数；`NORMALIZER` 用于把密集子图得分转换为便于比较的归一化指标。

In [ ]:
TARGET_SIZE = 8
ITERATIONS = 500
NORMALIZER = TARGET_SIZE * (TARGET_SIZE - 1)
RANDOM_SEED = 7

## 3. 加载图数据并查看结构

In [ ]:
adjacency = load_planted_dense_graph()
print('Graph shape:', adjacency.shape)

ax = draw_graph(adjacency, title='Planted dense graph')
plt.show()

## 4. 贪婪删除基线

贪婪删除每次移除一个节点，保留当前目标函数最高的候选子图。

In [ ]:
greedy_result = greedy_dense_subgraph(adjacency, size=TARGET_SIZE)
print('Selected nodes:', greedy_result.nodes.tolist())
print('Normalized score:', greedy_result.best_score / NORMALIZER)

draw_graph(adjacency, greedy_result.nodes, title='Greedy dense subgraph')
plt.show()

## 5. 随机搜索与模拟退火

随机搜索均匀抽取固定大小子图；模拟退火在接受更优解的同时，允许按温度接受少量较差候选，从而增加搜索空间探索能力。

In [ ]:
random_result = random_dense_search(
    adjacency,
    size=TARGET_SIZE,
    iterations=ITERATIONS,
    seed=RANDOM_SEED,
)
annealing_result = simulated_annealing_dense_search(
    adjacency,
    size=TARGET_SIZE,
    iterations=ITERATIONS,
    change_count=TARGET_SIZE,
    temperature=0.1,
    cooling_ratio=0.995,
    seed=RANDOM_SEED,
)

print('Random search nodes:', random_result.nodes.tolist())
print('Random normalized score:', random_result.best_score / NORMALIZER)
print('Annealing nodes:', annealing_result.nodes.tolist())
print('Annealing normalized score:', annealing_result.best_score / NORMALIZER)

## 6. 使用本地采样数据库增强搜索

示例采样数据库由 SDK 读取为候选子图集合，非零模式会被解释为候选节点。

In [ ]:
sample_database = load_sample_database('GBS_samples.npy')
sample_result = sample_database_search(
    adjacency,
    sample_database,
    size=TARGET_SIZE,
    iterations=ITERATIONS,
    seed=RANDOM_SEED,
)

print('Sample-enhanced nodes:', sample_result.nodes.tolist())
print('Sample-enhanced normalized score:', sample_result.best_score / NORMALIZER)

draw_graph(adjacency, sample_result.nodes, title='Sample-enhanced dense subgraph')
plt.show()

## 7. 对比收敛曲线

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(random_result.scores / NORMALIZER, label='Random Search')
plt.plot(annealing_result.scores / NORMALIZER, label='Annealing')
plt.plot(sample_result.scores / NORMALIZER, label='Sample Enhanced')
plt.axhline(greedy_result.best_score / NORMALIZER, linestyle='--', label='Greedy')
plt.xlabel('Iteration')
plt.ylabel('Normalized dense score')
plt.legend()
plt.tight_layout()
plt.show()